In [ ]:
import pandas as pd

# Load just the first 5 rows to test the connection instantly
df = pd.read_csv('swift_umdac_datathon_2026_fictitious_payment_data.csv', nrows=5)

# Display the data
print(df)

In [ ]:
import pandas as pd

# 1. Load the ENTIRE dataset 
df = pd.read_csv('swift_umdac_datathon_2026_fictitious_payment_data.csv')

# 2. Check for missing data (Required for the Datathon Issues Log)
print("--- Missing Values ---")
print(df.isnull().sum())

# 3. Convert the epoch timestamps to readable dates
df['datetime'] = pd.to_datetime(df['EPOCH_SECONDS'], unit='s')

# 4. Count exactly how many of each anomaly exist in the dataset
print("\n--- Anomaly Counts ---")
print(df['ANOMALY_TYPE'].value_counts())

In [ ]:
# 1. Drop the useless disclaimer column
df = df.drop(columns=['DISCLAIMER'])

# 2. Separate the anomalies from the normal baseline data
normal_txns = df[df['ANOMALY_TYPE'].isna()]
anomalies = df[df['ANOMALY_TYPE'].notna()]

# 3. Create the FAT_FINGER_MISTAKE dataset for Claude
fat_finger = anomalies[anomalies['ANOMALY_TYPE'] == 'FAT_FINGER_MISTAKE']

# To find the exact mathematical multiplier for a fat finger mistake, 
# Claude needs the normal payment history for the specific accounts that made the mistake.
ff_accounts = fat_finger['ORDERING_ACCOUNT'].unique()
ff_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(ff_accounts)]

# Combine them and export to a small CSV
fat_finger_export = pd.concat([fat_finger, ff_history])
fat_finger_export.to_csv('claude_fat_finger.csv', index=False)

print(f"Success! Exported {len(fat_finger_export)} rows to 'claude_fat_finger.csv'")

In [ ]:
import pandas as pd

# 1. Isolate the Fat Finger anomalies
fat_finger = anomalies[anomalies['ANOMALY_TYPE'] == 'FAT_FINGER_MISTAKE']
ff_accounts = fat_finger['ORDERING_ACCOUNT'].unique()

# 2. Get the normal history for these accounts
ff_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(ff_accounts)]

# 3. Calculate the MEDIAN normal amount for each account locally!
ff_medians = ff_history.groupby('ORDERING_ACCOUNT')['USD_AMOUNT'].median().reset_index()
ff_medians.rename(columns={'USD_AMOUNT': 'MEDIAN_NORMAL_AMOUNT'}, inplace=True)

# 4. Attach the median right next to the anomaly row
fat_finger_mini = pd.merge(fat_finger, ff_medians, on='ORDERING_ACCOUNT', how='left')

# 5. Export this tiny, summarized file
fat_finger_mini.to_csv('claude_fat_finger_mini.csv', index=False)

print(f"Success! Exported {len(fat_finger_mini)} rows to 'claude_fat_finger_mini.csv'. It should be tiny now!")

In [ ]:
# 1. Grab the specific sender & receiver pairs from the Fat Finger anomalies
ff_pairs = fat_finger[['ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT']].drop_duplicates()

# 2. Find the exact matching normal transactions for these specific pairs in the 10-million row dataset
exact_history = pd.merge(normal_txns, ff_pairs, on=['ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT'], how='inner')

# 3. Combine the anomalies and their exact normal history
exact_match_export = pd.concat([fat_finger, exact_history])

# 4. Keep only the essential columns so the file size is incredibly small
columns_to_keep = ['ANOMALY_TYPE', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 'USD_AMOUNT', 'CURRENCY_CODE']
exact_match_tiny = exact_match_export[columns_to_keep]

# 5. Export for Claude
exact_match_tiny.to_csv('claude_fat_finger_exact_matches.csv', index=False)

print(f"Success! Exported {len(exact_match_tiny)} rows to 'claude_fat_finger_exact_matches.csv'.")

In [ ]:
# 1. Isolate the Duplicate anomalies
duplicates = anomalies[anomalies['ANOMALY_TYPE'] == 'DUPLICATE_PAYMENT']

# 2. Grab the sender & receiver pairs involved in these duplicates
dup_pairs = duplicates[['ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT']].drop_duplicates()

# 3. Pull their normal history so Claude can find the original transaction timestamp
dup_history = pd.merge(normal_txns, dup_pairs, on=['ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT'], how='inner')

# 4. Combine and keep the crucial time and amount columns
dup_export = pd.concat([duplicates, dup_history])
columns_to_keep = ['ANOMALY_TYPE', 'EPOCH_SECONDS', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 'USD_AMOUNT', 'CURRENCY_CODE']
dup_tiny = dup_export[columns_to_keep]

# 5. Export for Claude
dup_tiny.to_csv('claude_duplicates.csv', index=False)

print(f"Success! Exported {len(dup_tiny)} rows to 'claude_duplicates.csv'")

In [ ]:
# 1. Isolate the Account Replacement Scam anomalies
scam = anomalies[anomalies['ANOMALY_TYPE'] == 'ACCOUNT_REPLACEMENT_SCAM']

# 2. Get the specific ordering accounts (the victims)
scam_accounts = scam['ORDERING_ACCOUNT'].unique()

# 3. Pull their normal payment history to see who they USUALLY pay
scam_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(scam_accounts)]

# 4. Combine and keep the essential columns for pattern matching
scam_export = pd.concat([scam, scam_history])
columns_to_keep = ['ANOMALY_TYPE', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 'BENEFICIARY_BIC8', 'USD_AMOUNT', 'EPOCH_SECONDS']
scam_tiny = scam_export[columns_to_keep]

# 5. Export for Claude
scam_tiny.to_csv('claude_scam.csv', index=False)

print(f"Success! Exported {len(scam_tiny)} rows to 'claude_scam.csv'")

In [ ]:
# 1. Isolate the Customer Recall anomalies
recall = anomalies[anomalies['ANOMALY_TYPE'] == 'CUSTOMER_RECALL']

# 2. Get the specific ordering accounts involved
recall_accounts = recall['ORDERING_ACCOUNT'].unique()

# 3. Pull their normal payment history to see what they are recalling
recall_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(recall_accounts)]

# 4. Combine and keep the essential columns
recall_export = pd.concat([recall, recall_history])
columns_to_keep = ['ANOMALY_TYPE', 'EPOCH_SECONDS', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 'USD_AMOUNT', 'TRANSACTION_TYPE_CODE', 'CURRENCY_CODE']
recall_tiny = recall_export[columns_to_keep]

# 5. Export for Claude
recall_tiny.to_csv('claude_recall.csv', index=False)

print(f"Success! Exported {len(recall_tiny)} rows to 'claude_recall.csv'")

In [ ]:
# 1. Combine the recall anomalies and their normal history again
recall_export_details = pd.concat([recall, recall_history])

# 2. Keep the specific processing and routing columns Claude requested
columns_to_keep_details = [
    'ANOMALY_TYPE', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 
    'TOTAL_DURATION', 'BIC8_PATH', 'CONTINUITY_QFLAG', 
    'PARTICIPANT_COUNT', 'CURRENCY_CONVERSION_COUNT', 
    'ORDERING_TYPE', 'BENEFICIARY_TYPE'
]
recall_details_tiny = recall_export_details[columns_to_keep_details]

# 3. Export this new detailed slice
recall_details_tiny.to_csv('claude_recall_details.csv', index=False)

print(f"Success! Exported {len(recall_details_tiny)} rows to 'claude_recall_details.csv'")

In [ ]:
# 1. Grab just a small sample (20 max) of normal history per account to keep the file tiny
recall_history_sampled = recall_history.groupby('ORDERING_ACCOUNT').head(20)

# 2. Combine the recall anomalies with this small normal sample
recall_export_text = pd.concat([recall, recall_history_sampled])

# 3. Keep only the final, untested text and token columns
columns_to_keep_text = [
    'ANOMALY_TYPE', 'ORDERING_ACCOUNT', 
    'ORDERING_DETAILS', 'ORDERING_TOKENS', 
    'BENEFICIARY_DETAILS', 'BENEFICIARY_TOKENS'
]
recall_text_tiny = recall_export_text[columns_to_keep_text]

# 4. Export for Claude
recall_text_tiny.to_csv('claude_recall_text.csv', index=False)

print(f"Success! Exported {len(recall_text_tiny)} rows to 'claude_recall_text.csv'")

In [ ]:
import ast

# 1. Reveal ALL anomalies without truncation
print("--- FULL ANOMALY COUNTS ---")
print(df['ANOMALY_TYPE'].value_counts(dropna=False).to_string())

# 2. Parse the TOKENS from strings into actual Python lists (Cleaning step)
# (Using a small sample printout so it doesn't crash your memory)
print("\n--- Parsing TOKENS ---")
df['ORDERING_TOKENS_PARSED'] = df['ORDERING_TOKENS'].fillna('[]').apply(ast.literal_eval)
df['BENEFICIARY_TOKENS_PARSED'] = df['BENEFICIARY_TOKENS'].fillna('[]').apply(ast.literal_eval)
print("Tokens successfully parsed into lists!")

In [ ]:
# 1. Isolate the CEO Scam anomalies
ceo = anomalies[anomalies['ANOMALY_TYPE'] == 'CEO_SCAM']

# 2. Get the victims' ordering accounts
ceo_accounts = ceo['ORDERING_ACCOUNT'].unique()

# 3. Pull their normal history so Claude can see if the amount or beneficiary is unusual
ceo_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(ceo_accounts)]

# 4. Combine and keep the essential columns (including text details for urgent messages)
ceo_export = pd.concat([ceo, ceo_history])
columns_to_keep_ceo = [
    'ANOMALY_TYPE', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 
    'USD_AMOUNT', 'ORDERING_DETAILS', 'BENEFICIARY_DETAILS'
]
ceo_tiny = ceo_export[columns_to_keep_ceo]

# 5. Export for Claude
ceo_tiny.to_csv('claude_ceo_scam.csv', index=False)

print(f"Success! Exported {len(ceo_tiny)} rows to 'claude_ceo_scam.csv'")

In [ ]:
# 1. Isolate the Romance Scam anomalies
romance = anomalies[anomalies['ANOMALY_TYPE'] == 'ROMANCE_SCAM']

# 2. Get the victims' ordering accounts
romance_accounts = romance['ORDERING_ACCOUNT'].unique()

# 3. Pull their normal history
romance_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(romance_accounts)]

# 4. Combine and keep the essential columns to track repeated payments over time
romance_export = pd.concat([romance, romance_history])
columns_to_keep_romance = [
    'ANOMALY_TYPE', 'EPOCH_SECONDS', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 
    'USD_AMOUNT', 'ORDERING_TYPE', 'BENEFICIARY_TYPE'
]
romance_tiny = romance_export[columns_to_keep_romance]

# 5. Export for Claude
romance_tiny.to_csv('claude_romance_scam.csv', index=False)

print(f"Success! Exported {len(romance_tiny)} rows to 'claude_romance_scam.csv'")

In [ ]:
# Re-establish the baseline variables just in case the notebook forgot them
normal_txns = df[df['ANOMALY_TYPE'].isna()]
anomalies = df[df['ANOMALY_TYPE'].notna()]

# 1. Isolate the Romance Scam anomalies
romance = anomalies[anomalies['ANOMALY_TYPE'] == 'ROMANCE_SCAM']

# 2. Get the victims' ordering accounts
romance_accounts = romance['ORDERING_ACCOUNT'].unique()

# 3. Pull their normal history
romance_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(romance_accounts)]

# 4. Combine and keep the essential columns to track repeated payments over time
romance_export = pd.concat([romance, romance_history])
columns_to_keep_romance = [
    'ANOMALY_TYPE', 'EPOCH_SECONDS', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 
    'USD_AMOUNT', 'ORDERING_TYPE', 'BENEFICIARY_TYPE'
]
romance_tiny = romance_export[columns_to_keep_romance]

# 5. Export for Claude
romance_tiny.to_csv('claude_romance_scam.csv', index=False)

print(f"Success! Exported {len(romance_tiny)} rows to 'claude_romance_scam.csv'")

In [ ]:
import pandas as pd

# 1. Reload the entire dataset (Wait 2-4 minutes for this to finish)
df = pd.read_csv('swift_umdac_datathon_2026_fictitious_payment_data.csv', low_memory=False)

# 2. Re-establish the baseline variables
normal_txns = df[df['ANOMALY_TYPE'].isna()]
anomalies = df[df['ANOMALY_TYPE'].notna()]

# 3. Isolate the Romance Scam anomalies
romance = anomalies[anomalies['ANOMALY_TYPE'] == 'ROMANCE_SCAM']
romance_accounts = romance['ORDERING_ACCOUNT'].unique()

# 4. Pull their normal history
romance_history = normal_txns[normal_txns['ORDERING_ACCOUNT'].isin(romance_accounts)]

# 5. Combine and keep the essential columns to track repeated payments over time
romance_export = pd.concat([romance, romance_history])
columns_to_keep_romance = [
    'ANOMALY_TYPE', 'EPOCH_SECONDS', 'ORDERING_ACCOUNT', 'BENEFICIARY_ACCOUNT', 
    'USD_AMOUNT', 'ORDERING_TYPE', 'BENEFICIARY_TYPE'
]
romance_tiny = romance_export[columns_to_keep_romance]

# 6. Export for Claude
romance_tiny.to_csv('claude_romance_scam.csv', index=False)

print(f"Success! Exported {len(romance_tiny)} rows to 'claude_romance_scam.csv'")